In [1]:
# --- Imports and path constants ---
import os
import polars as pl
import pandas as pd
import anndata as ad
import matplotlib.pyplot as plt

COMBINED_ROOT = "/scratch/combined_datasets"
H5AD_ROOT     = "/scratch/h5ad_outputs"


# wnm_exc_vis_manuscript
data from Matt Malllory

Three CSVs: morphology features (`AxonRawReatureWide`), metadata + MET labels (`FullMorphMetaData_Master`), and brain-region projection matrix (`ProjectionMatrix`).

In [2]:
CSV_DATA_ROOT = "/root/capsule/data"


def read_csv_labels(
    path: str,
    id_col: str,
    label_col: str,
    rename_label: str = "predicted_label",
    strip_swc: bool = False,
    id_is_int: bool = False,
) -> pd.DataFrame:
    """Read a raw CSV into an (id, predicted_label) DataFrame.

    Parameters
    ----------
    path : str         Path to the CSV file.
    id_col : str       Column to use as cell ID.
    label_col : str    Column to use as the predicted label.
    rename_label : str Output column name for the label (default 'predicted_label').
    strip_swc : bool   If True, strip a trailing '.swc' from ID strings.
    id_is_int : bool   If True, cast id to int64.
    """
    df = pd.read_csv(path, usecols=[id_col, label_col])
    df = df.rename(columns={id_col: "id", label_col: rename_label})
    if strip_swc:
        df["id"] = df["id"].str.removesuffix(".swc")
    if id_is_int:
        df["id"] = df["id"].astype("int64")
    return df.reset_index(drop=True)


def read_csv_feats(
    path: str,
    id_col: str,
    drop_cols: tuple | list = (),
    strip_swc: bool = False,
    id_is_int: bool = False,
) -> pd.DataFrame:
    """Read a raw CSV into an (id, *features) DataFrame.

    Parameters
    ----------
    path : str              Path to the CSV file.
    id_col : str            Column to use as cell ID (renamed to 'id').
    drop_cols : sequence    Non-ID columns to drop (label/metadata columns).
    strip_swc : bool        If True, strip a trailing '.swc' from ID strings.
    id_is_int : bool        If True, cast id to int64.
    """
    df = pd.read_csv(path, index_col=False)
    df = df.rename(columns={id_col: "id"})
    # Drop label/metadata columns and any leftover pandas auto-index columns
    to_drop = [c for c in drop_cols if c in df.columns]
    to_drop += [c for c in df.columns if c != "id" and str(c).startswith("Unnamed:")]
    df = df.drop(columns=to_drop)
    if strip_swc:
        df["id"] = df["id"].str.removesuffix(".swc")
    if id_is_int:
        df["id"] = df["id"].astype("int64")
    return df.reset_index(drop=True)


In [3]:
_WNM_ROOT = os.path.join(CSV_DATA_ROOT, "wnm_exc_vis_manuscript")

_wnm_axon_raw   = pd.read_csv(os.path.join(_WNM_ROOT, "AxonRawReatureWide.csv"))
_wnm_meta       = pd.read_csv(os.path.join(_WNM_ROOT, "FullMorphMetaData_Master.csv"))
_wnm_proj       = pd.read_csv(os.path.join(_WNM_ROOT, "ProjectionMatrix_tip_and_branch_roll_up.csv"))

print(f"AxonRawReatureWide        {_wnm_axon_raw.shape}")
display(_wnm_axon_raw.head(3))

print(f"\nFullMorphMetaData_Master  {_wnm_meta.shape}")
display(_wnm_meta.head(3))

print(f"\nProjectionMatrix (first 6 cols shown)  {_wnm_proj.shape}")
display(_wnm_proj.iloc[:3, :7])  # truncate — 221 columns total


AxonRawReatureWide        (345, 52)


,specimen_id,apical_dendrite_bias_x,apical_dendrite_bias_y,apical_dendrite_depth_pc_0,apical_dendrite_depth_pc_1,apical_dendrite_depth_pc_2,apical_dendrite_depth_pc_3,apical_dendrite_depth_pc_4,apical_dendrite_early_branch_path,apical_dendrite_extent_x,...,axon_max_branch_order,axon_max_euclidean_distance,axon_max_path_distance,axon_mean_contraction,axon_num_branches,axon_soma_percentile_x,axon_soma_percentile_y,axon_total_length,soma_aligned_dist_from_pia,soma_surface_area
0,17109_6201-X4328-Y6753_reg,21.311538,2.306327,-446.959647,-77.960333,-116.411504,-89.641313,-76.897975,0.479785,291.778779,...,18.0,782.489366,2295.547989,0.775924,133.0,0.456671,0.291300,32679.601463,755.648634,0.0
1,17109_6301-X4756-Y24516_reg,39.591795,17.363311,-456.398402,-70.969916,-123.361715,-92.973863,-99.533545,0.484623,255.485239,...,11.0,798.003108,1794.220795,0.789746,83.0,0.434223,0.193180,24944.239274,779.803826,0.0
2,17109_6601-X4384-Y7436_reg,110.472066,-7.655321,-447.486796,-103.720271,-125.564660,-117.194281,-51.706742,0.498860,331.019359,...,10.0,722.175168,2298.330681,0.773319,79.0,0.297082,0.511213,20598.298943,727.831495,0.0



FullMorphMetaData_Master  (341, 17)


,Unnamed: 0,predicted_met_type,probability,ccf_soma_location,ccf_soma_location_nolayer,ccf_soma_x,ccf_soma_y,ccf_soma_z,distance_soma_moved_out_of_brain_correction,cre_line,azimuth,altitude,auto_projection_subclass,dend_derived_predicted_subclass,dend_derived_predicted_probability,local_axon_derived_subclass,met_classifier_routing_call
0,182709_6984-X2452-Y12423_reg.swc,L5 ET-2,0.988,VISpm5,VISpm,8899.823,643.262,4324.473,37.416574,Ai82;Ai139_375886-182709,35.212157,3.519493,ET,ET,0.760219,NaN,ET
1,182709_7126-X2913-Y10535_reg.swc,L5 ET-3,0.918,VISp5,VISp,9117.453,1064.075,3550.508,24.494897,Ai82;Ai139_375886-182709,48.447344,-0.296991,ET,ET,0.949430,NaN,ET
2,182724_5937-X3804-Y11955_reg.swc,L5 ET-2,0.724,VISa5,VISa,7168.289,953.689,3959.651,42.426407,Fezf2-CreER;Ai166_405426-182724,41.861469,1.670020,ET,ET,0.928120,NaN,ET



ProjectionMatrix (first 6 cols shown)  (345, 221)


,Unnamed: 0,ipsi_VISam,ipsi_VISp,ipsi_VISpm,ipsi_VISrl,contra_VISpor,ipsi_CP
0,18864_6734-X4899-Y27447_reg.swc,8287.70664,34450.175934,483.223644,5737.760785,0.000000,0.000000
1,191812_7938-X6892-Y25312_reg.swc,0.00000,794.102517,0.000000,0.000000,1045.437572,9243.339922
2,211550_7718-X19461-Y16950_reg.swc,0.00000,6473.751624,0.000000,0.000000,0.000000,0.000000


# EM_exc_mettypes

`EM_RFC_MET_Predictions_FCNormCorrect_And_Uncorrect.csv` — EM EXC cells with RFC-predicted MET types (corrected and uncorrected). This CSV contains only labels (no morphological features).

In [4]:
_EM_EXC_ROOT = os.path.join(CSV_DATA_ROOT, "EM_exc_mettypes")

_em_exc_met = pd.read_csv(os.path.join(
    _EM_EXC_ROOT, "EM_RFC_MET_Predictions_FCNormCorrect_And_Uncorrect.csv"))
print(f"EM_RFC_MET_Predictions  {_em_exc_met.shape}")
display(_em_exc_met.head(3))


EM_RFC_MET_Predictions  (38267, 7)


,Unnamed: 0,id,uncorrected_rfc_predicted_met_type,uncorrected_rfc_probability,corrected_rfc_predicted_met_type,corrected_rfc_probability,labels_agree
0,0,26468,IT-MET-1,0.868,IT-MET-2,0.978,False
1,1,26554,IT-MET-1,1.000,IT-MET-2,0.948,False
2,2,26556,IT-MET-1,0.994,IT-MET-2,0.948,False


# The visp met taxonomy

In [6]:
# --- VISp EXC MET type taxonomy from cluster schema ---
df_visp_met_taxonomy = (
    pl.read_delta(os.path.join(COMBINED_ROOT, "cluster"))
    .filter(pl.col("project_id") == "visp_met_types")
    .sort(["level", "id"])
)

# Leaf-level EXC (Glutamatergic) MET types
df_visp_exc_met = df_visp_met_taxonomy.filter(
    (pl.col("level") == 2) & (pl.col("parent") == "Glutamatergic")
)

print(f"Full visp_met_types taxonomy: {df_visp_met_taxonomy.shape}")
print(f"EXC MET leaf types          : {df_visp_exc_met.shape[0]} types")
display(df_visp_exc_met.select(["id", "parent", "hex_color", "heirachy_category"]))

Full visp_met_types taxonomy: (48, 9)
EXC MET leaf types          : 17 types


id,parent,hex_color,heirachy_category
str,str,str,str
"""L2/3 IT""","""Glutamatergic""","""#7AE6AB""","""cluster"""
"""L4 IT""","""Glutamatergic""","""#00979D""","""cluster"""
"""L4/L5 IT""","""Glutamatergic""","""#00DDC5""","""cluster"""
"""L5 ET-1 Chrna6""","""Glutamatergic""","""#0000FF""","""cluster"""
"""L5 ET-2""","""Glutamatergic""","""#22737F""","""cluster"""
…,…,…,…
"""L6 CT-2""","""Glutamatergic""","""#578EBF""","""cluster"""
"""L6 IT-1""","""Glutamatergic""","""#C2E32C""","""cluster"""
"""L6 IT-2""","""Glutamatergic""","""#96E32C""","""cluster"""


## sorted labels side by side

In [7]:
#EM
mm = list(_em_exc_met.corrected_rfc_predicted_met_type.unique())
mm.sort()
mm

['CT-MET-1',
 'IT-MET-1',
 'IT-MET-2',
 'IT-MET-3',
 'IT-MET-4',
 'IT-MET-5',
 'IT-MET-6',
 'IT-MET-7',
 'IT-MET-8',
 'L6b-MET-1',
 'L6b-MET-2',
 'L6b-MET-3',
 'NP-MET-1',
 'PT-MET-1',
 'PT-MET-2',
 'PT-MET-3']

In [9]:
# Matt wnm
mw = list(_wnm_meta.predicted_met_type.unique())
mw.sort()
mw

['L2/3 IT',
 'L4 IT',
 'L4/L5 IT',
 'L5 ET-1 Chrna6',
 'L5 ET-2',
 'L5 ET-3',
 'L5 IT-2',
 'L5 NP',
 'L5/L6 IT Car3',
 'L6 CT-1',
 'L6 CT-2',
 'L6 IT-1',
 'L6 IT-2',
 'L6 IT-3',
 'L6b']

In [10]:
# Nathan wnm
vv = list(df_visp_exc_met['id'].unique())
vv.sort()
vv

['L2/3 IT',
 'L4 IT',
 'L4/L5 IT',
 'L5 ET-1 Chrna6',
 'L5 ET-2',
 'L5 ET-3',
 'L5 IT-1',
 'L5 IT-2',
 'L5 IT-3 Pld5',
 'L5 NP',
 'L5/L6 IT Car3',
 'L6 CT-1',
 'L6 CT-2',
 'L6 IT-1',
 'L6 IT-2',
 'L6 IT-3',
 'L6b']

In [11]:
# claude mapping based on the manuscript descriptive names
met_to_descriptive = {
    # IT types (8)
    'IT-MET-1': 'L2/3 IT',
    'IT-MET-2': 'L4 IT',
    'IT-MET-3': 'L4/L5 IT',
    'IT-MET-4': 'L5 IT-1',
    'IT-MET-5': 'L5 IT-2 Pld5',
    'IT-MET-6': 'L5/L6 IT Car3',
    'IT-MET-7': 'L6 IT-1',
    'IT-MET-8': 'L6 IT-2',
    # PT/ET types (3)
    'PT-MET-1': 'L5 ET-1 Chrna6',
    'PT-MET-2': 'L5 ET-2',
    'PT-MET-3': 'L5 ET-3 Stac',
    # NP type (1)
    'NP-MET-1': 'L5 NP',
    # CT type (1)
    'CT-MET-1': 'L6 CT',
    # L6b types (3)
    'L6b-MET-1': 'L6b-1',
    'L6b-MET-2': 'L6b-2 Ngf',
    'L6b-MET-3': 'L6b-3',
}

# claude mapping to the existing vis names
met_to_second_list = {
    # IT types — high confidence
    'IT-MET-1': 'L2/3 IT',
    'IT-MET-2': 'L4 IT',
    'IT-MET-3': 'L4/L5 IT',
    'IT-MET-4': 'L5 IT-1',
    'IT-MET-5': 'L5 IT-2',
    'IT-MET-6': 'L5 IT-3 Pld5',
    'IT-MET-7': 'L5/L6 IT Car3',
    'IT-MET-8': 'L6 IT-1',        # paper has L6 IT-1 and L6 IT-2;
                                    # second list has L6 IT-1, L6 IT-2, L6 IT-3
                                    # so one L6 IT in the second list has no MET match

    # PT/ET types — high confidence (PT = pyramidal tract = ET = extratelencephalic)
    'PT-MET-1': 'L5 ET-1 Chrna6',
    'PT-MET-2': 'L5 ET-2',
    'PT-MET-3': 'L5 ET-3',

    # NP — trivial
    'NP-MET-1': 'L5 NP',

    # CT — paper has 1, second list has 2
    'CT-MET-1': 'L6 CT-1',        # L6 CT-2 in second list has no MET match

    # L6b — paper has 3 subtypes, second list collapses to 1
    'L6b-MET-1': 'L6b',
    'L6b-MET-2': 'L6b',           # all three merge into single "L6b"
    'L6b-MET-3': 'L6b',
}